<a href="https://colab.research.google.com/github/priyapriya1342006-oss/Gen-AI-Mastery/blob/main/CRUD_operations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install -q chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [2]:
import chromadb

# In-memory client (resets when runtime restarts) - great for learning
client = chromadb.Client()

# Create a fresh collection to practice CRUD operations on
collection = client.create_collection(name="crud_demo")

print("Collection created:", collection.name)
print("Current document count:", collection.count())

Collection created: crud_demo
Current document count: 0


In [3]:
collection.add(
    documents=[
        "Python is a popular programming language for AI.",
        "The weather today is sunny and warm.",
        "Machine learning models learn patterns from data.",
        "I enjoy playing football on weekends.",
        "Deep learning uses neural networks with many layers."
    ],
    metadatas=[
        {"category": "tech"},
        {"category": "weather"},
        {"category": "tech"},
        {"category": "sports"},
        {"category": "tech"}
    ],
    ids=["doc1", "doc2", "doc3", "doc4", "doc5"]
)

print("Documents added! Total count:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 41.5MiB/s]


Documents added! Total count: 5


In [4]:
preview = collection.peek(limit=3)

for doc_id, doc in zip(preview["ids"], preview["documents"]):
    print(doc_id, "->", doc)

doc1 -> Python is a popular programming language for AI.
doc2 -> The weather today is sunny and warm.
doc3 -> Machine learning models learn patterns from data.


In [5]:
result = collection.get(ids=["doc1", "doc3"])

for doc_id, doc, meta in zip(result["ids"], result["documents"], result["metadatas"]):
    print(f"{doc_id} | {meta} | {doc}")

doc1 | {'category': 'tech'} | Python is a popular programming language for AI.
doc3 | {'category': 'tech'} | Machine learning models learn patterns from data.


In [6]:
results = collection.query(
    query_texts=["Tell me about AI and neural networks"],
    n_results=2
)

for doc_id, doc, distance in zip(results["ids"][0], results["documents"][0], results["distances"][0]):
    print(f"{doc_id} | distance={distance:.4f} | {doc}")

doc5 | distance=0.9317 | Deep learning uses neural networks with many layers.
doc1 | distance=0.9516 | Python is a popular programming language for AI.


In [7]:
results = collection.query(
    query_texts=["something fun to do outside"],
    n_results=3,
    where={"category": "sports"}
)

for doc_id, doc, distance in zip(results["ids"][0], results["documents"][0], results["distances"][0]):
    print(f"{doc_id} | distance={distance:.4f} | {doc}")

doc4 | distance=1.3895 | I enjoy playing football on weekends.


In [8]:
collection.update(
    ids=["doc4"],
    documents=["I enjoy playing football and basketball on weekends."],
    metadatas=[{"category": "sports", "updated": True}]
)

updated = collection.get(ids=["doc4"])
print(updated["documents"])
print(updated["metadatas"])

['I enjoy playing football and basketball on weekends.']
[{'category': 'sports', 'updated': True}]


In [9]:
collection.upsert(
    documents=[
        "Deep learning uses neural networks with many layers and GPUs.",  # existing id -> will update
        "The Eiffel Tower is located in Paris."                           # new id -> will create
    ],
    ids=["doc5", "doc6"]
)

print("Total count after upsert:", collection.count())
print(collection.get(ids=["doc5", "doc6"])["documents"])

Total count after upsert: 6
['Deep learning uses neural networks with many layers and GPUs.', 'The Eiffel Tower is located in Paris.']


In [10]:
collection.delete(ids=["doc2"])

print("Total count after delete:", collection.count())
print("Remaining ids:", collection.get()["ids"])

Total count after delete: 5
Remaining ids: ['doc1', 'doc3', 'doc4', 'doc5', 'doc6']


In [11]:
collection.delete(where={"category": "tech"})

print("Total count after category delete:", collection.count())
print("Remaining ids:", collection.get()["ids"])

Total count after category delete: 2
Remaining ids: ['doc4', 'doc6']


In [12]:

final_state = collection.get()

print("Final document count:", collection.count())
for doc_id, doc, meta in zip(final_state["ids"], final_state["documents"], final_state["metadatas"]):
    print(f"{doc_id} | {meta} | {doc}")

Final document count: 2
doc4 | {'updated': True, 'category': 'sports'} | I enjoy playing football and basketball on weekends.
doc6 | None | The Eiffel Tower is located in Paris.
